# Aula 7 — O que dá para afirmar sobre um modelo que eu não consigo ler?

**Disciplina 2 · Aprendizado de Máquina · IASEG**

Na Aula 5, a floresta ganhou nos números, e tem quase 10 mil folhas: não dá para ler. Hoje, três perguntas sobre um modelo assim:

- **Parte 1** — O que a floresta usou? (embaralhar uma coluna)
- **Parte 2** — Duas colunas que dizem quase a mesma coisa
- **Parte 3** — Por que ela? Um caso como uma soma (SHAP)
- **Parte 4** — O banco sem a idade: tirar a coluna tira a informação?
- **A célula para o projeto de vocês** — o que o modelo de vocês usou (pergunta 6 do banco)

## Configuração

`pandas` para os dados, `numpy` para as contas. O resto é importado na etapa em que for usado.

In [1]:
import pandas as pd
import numpy as np

---

# Parte 1 — O que a floresta usou?

## Etapa 1 — A floresta da Aula 5, de novo

O mesmo Titanic, as mesmas seis colunas, a mesma divisão e a mesma floresta da Aula 5.

- `test_size=0.3`: 30% dos passageiros ficam guardados para o teste.
- `random_state=42`: fixa o sorteio da divisão (e o das árvores), para todo mundo ter os mesmos números (qualquer número serviria).
- `stratify=y`: treino e teste ficam com a mesma proporção de sobreviventes, para a acurácia do teste não depender de um sorteio desequilibrado.

In [23]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
COLUNAS = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare']

df = pd.read_csv(URL)

# As seis colunas, a resposta e uma coluna extra para a Parte 2; tira quem não tem idade
dados = df[COLUNAS + ['adult_male', 'survived']].dropna().copy()

# sex vira 0/1 (1 = mulher, 0 = homem); adult_male vira 0/1 (1 = homem adulto)
dados['sex'] = (dados['sex'] == 'female').astype(int)
dados['adult_male'] = dados['adult_male'].astype(int)

X = dados[COLUNAS]
y = dados['survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

floresta = RandomForestClassifier(n_estimators=100, random_state=42)
floresta.fit(X_train, y_train)

acuracia = floresta.score(X_test, y_test)
print(f'Floresta: acurácia no teste {acuracia:.1%}')

Floresta: acurácia no teste 79.5%


## Etapa 2 — Embaralhar uma coluna, à mão

A ideia do quadro: **embaralhar uma coluna do teste** apaga o que ela sabe sobre cada passageiro, sem mexer em mais nada.
O modelo **não é treinado de novo**: é a mesma floresta, só respondendo outra vez.
Se ela dependia da coluna, a acurácia no teste cai. Se não dependia, a acurácia fica onde estava.

(O nome em inglês, e o da função que vamos usar: *permutation importance*.)

Escolham **uma** das seis colunas (`'pclass'`, `'sex'`, `'age'`, `'sibsp'`, `'parch'`, `'fare'`).

**Antes de rodar:** quanto vocês acham que a acurácia vai cair?

In [16]:
# A coluna que vamos embaralhar: escolham uma das seis (entre aspas, como na lista acima)
coluna = "sex"

# Uma cópia do teste, com essa coluna numa ordem sorteada (o resto fica igual)
X_embaralhado = X_test.copy()
X_embaralhado[coluna] = X_test[coluna].sample(frac=1, random_state=1).values

acuracia_embaralhada = floresta.score(X_embaralhado, y_test)
print(f'Acurácia no teste, original: {acuracia:.1%}')
print(f'Com {coluna} embaralhada: {acuracia_embaralhada:.1%}')
print(f'Perdeu {100 * (acuracia - acuracia_embaralhada):.1f} pontos')

Acurácia no teste, original: 79.5%
Com sex embaralhada: 61.4%
Perdeu 18.1 pontos


Troquem a coluna e rodem de novo. Comparem com o que os colegas encontraram.

Um embaralhamento é **um sorteio**: com outra ordem, a perda seria um pouco diferente.
Por isso a biblioteca embaralha várias vezes e tira a média (próxima etapa).

## Etapa 3 — Todas as colunas, vinte vezes cada

`permutation_importance` faz a Etapa 2 para todas as colunas, `n_repeats=20` vezes cada, e devolve a perda média.

- `random_state=42`: fixa os sorteios dos embaralhamentos (qualquer número serviria).

**Antes de rodar:** ordenem as seis colunas, da que a floresta mais usou para a que menos usou.

In [27]:
from sklearn.inspection import permutation_importance

resultado = permutation_importance(floresta, X_test, y_test, n_repeats=20, random_state=42)


# Em pontos de acurácia: quanto a acurácia do teste cai, em média, quando a coluna é embaralhada
importancia = pd.DataFrame({
    'perde (pontos)': (100 * resultado.importances_mean).round(1),
    'pior vez': (100 * resultado.importances.max(axis=1)).round(1),
    'melhor vez': (100 * resultado.importances.min(axis=1)).round(1),
}, index=X_test.columns).sort_values('perde (pontos)', ascending=False)
# importancia

Isso responde à **pergunta 6** do banco (*quais variáveis o modelo de vocês mais usou?*), para qualquer modelo, sem precisar ler as árvores.

O que isso **não** responde: *por quê*. Uma coluna muito usada não é, por isso, uma causa (a tarifa salvou alguém?).

---

# Parte 2 — Duas colunas que dizem quase a mesma coisa

## Etapa 4 — Mais uma coluna: `adult_male`

O Titanic tem uma coluna `adult_male`: 1 para homem adulto, 0 para mulheres e meninos.
Ela diz quase o mesmo que `sex` (colinearidade, Aula 6).

Vamos treinar a mesma floresta com as sete colunas e medir de novo.

**Antes de rodar:** o que vai acontecer com a importância de `sex`?

In [ ]:
COLUNAS_7 = COLUNAS + ['adult_male']

# A mesma divisão: mesmo random_state, mesmo stratify, as mesmas linhas no teste
X7_train, X7_test, _, _ = train_test_split(dados[COLUNAS_7], y, test_size=0.3, random_state=42, stratify=y)

floresta_7 = RandomForestClassifier(n_estimators=100, random_state=42)
floresta_7.fit(X7_train, y_train)
print(f'Floresta com adult_male: teste {floresta_7.score(X7_test, y_test):.1%}')

resultado_7 = permutation_importance(floresta_7, X7_test, y_test, n_repeats=20, random_state=42)
pd.Series((100 * resultado_7.importances_mean).round(1), index=COLUNAS_7, name='perde (pontos)').sort_values(ascending=False)

O modelo **parou de usar o sexo**? Olhem de novo a tabela: a informação continua lá, só mudou de coluna.

Quando duas colunas dizem o mesmo, embaralhar uma só quase não dói, porque a outra ainda sabe.
A importância **se divide** entre as duas, e nenhuma delas parece importante sozinha.

## Etapa 5 — E se a gente tirar o sexo?

Tirem a coluna `sex` e treinem de novo, mantendo `adult_male`
(Aula 3: `03_Avaliacao…`, `drop(columns=...)`).

**Antes de rodar:** a acurácia vai cair muito, pouco, ou nada?

In [ ]:
# Tira a coluna sex (e fica com adult_male)
X_sem_sexo_train = X7_train.drop(columns=____)
X_sem_sexo_test = X7_test.drop(columns=____)

floresta_sem_sexo = RandomForestClassifier(n_estimators=100, random_state=42)
floresta_sem_sexo.fit(X_sem_sexo_train, y_train)
print(f'Sem sex, com adult_male: teste {floresta_sem_sexo.score(X_sem_sexo_test, y_test):.1%}')

Agora tirem **as duas** (`sex` e `adult_male`) e comparem.

In [ ]:
X_sem_as_duas_train = X7_train.drop(columns=['sex', 'adult_male'])
X_sem_as_duas_test = X7_test.drop(columns=['sex', 'adult_male'])

floresta_sem_as_duas = RandomForestClassifier(n_estimators=100, random_state=42)
floresta_sem_as_duas.fit(X_sem_as_duas_train, y_train)
print(f'Sem sex e sem adult_male: teste {floresta_sem_as_duas.score(X_sem_as_duas_test, y_test):.1%}')

**Tirar uma coluna não tira a informação**, se outra coluna carrega a mesma coisa.
Uma coluna que carrega outra se chama **proxy** (*uma variável que representa outra*).
Aqui o proxy é óbvio (o nome já diz "homem adulto"). Na Parte 4, ele não é.

---

# Parte 3 — Por que ela? Um caso como uma soma

A importância (Parte 1) fala do modelo **no geral**. Para **um caso**, existe o **SHAP**:
ele escreve a resposta da floresta para um passageiro como uma **soma**:
um **ponto de partida** (a chance média, sem saber nada do passageiro) **mais a mudança** que cada coluna provoca.
Cada mudança é a média de todas as ordens em que as colunas poderiam ser reveladas (o slide).

## Etapa 6 — A passageira G

A passageira do quadro, da Aula 4 e da Aula 5: mulher, 1ª classe, 31 anos.
A primeira linha instala a biblioteca `shap` (demora um pouco).
Depois, uma função que vamos usar nas três etapas: ela devolve a chance da floresta e a mudança de cada coluna.

In [ ]:
!pip install shap -q
import shap

explicador = shap.TreeExplainer(floresta)

def explica(caso):
    """A chance da floresta para um caso, e a mudança que cada coluna provoca (SHAP), de 0 a 100."""
    valores = explicador.shap_values(caso)
    # a biblioteca devolve os valores de 'morreu' e de 'sobreviveu'; ficamos com os de 'sobreviveu'
    # (o formato mudou entre versões do shap: a linha abaixo cobre os dois)
    valores = valores[1] if isinstance(valores, list) else valores[..., 1]
    partida = 100 * np.array(explicador.expected_value).ravel()[-1]
    mudancas = pd.Series(100 * np.ravel(valores), index=caso.columns).round(1)
    chance = 100 * floresta.predict_proba(caso)[0, 1]
    print(f'Ponto de partida (a chance média): {partida:.1f} de cada 100')
    print(mudancas.sort_values(key=abs, ascending=False).to_string())
    print(f'Soma: {partida:.1f} + ({mudancas.sum():.1f}) = {partida + mudancas.sum():.1f}')
    print(f'A floresta diz: {chance:.1f} de cada 100')

# A passageira G: mulher, 1ª classe, 31 anos (está no teste)
passageira = X_test[(X_test['sex'] == 1) & (X_test['pclass'] == 1) & (X_test['age'] == 31)]
print(passageira.to_string())
print()
explica(passageira)

Confiram: o ponto de partida mais as mudanças dá exatamente a chance da floresta?
Qual coluna mais mudou a chance dela? E qual puxou para baixo?

## Etapa 7 — E se ela fosse homem?

A mesma passageira, a mesma floresta (**não é treinada de novo**); só a coluna `sex` muda (Etapa 1: 1 = mulher, 0 = homem).

**Antes de rodar:** o que acontece com a chance? E com a mudança da coluna `sex`? As outras colunas continuam mudando a chance do mesmo jeito?

In [ ]:
# Uma cópia da passageira, com o sexo trocado para homem (Etapa 1: 1 = mulher, 0 = homem)
passageiro = passageira.copy()
passageiro['sex'] = ____

explica(passageiro)

Comparem as duas listas, coluna por coluna. Uma coluna que quase não mudava nada para ela pode mudar bastante para ele:
**a mudança de uma coluna depende das outras**. É por isso que o SHAP faz a média de todas as ordens.

E reparem: o SHAP diz **como esta resposta se divide** entre as colunas. Não diz que o sexo *causou* a sobrevivência: diz o que **o modelo** fez com ele.

## Etapa 8 — Um passageiro de vocês

Inventem um passageiro: troquem os valores abaixo (a classe, 1, 2 ou 3; o sexo, 1 = mulher, 0 = homem; a idade;
irmãos/cônjuge a bordo; pais/filhos a bordo; a tarifa, que na 3ª classe costuma ficar perto de 8 e na 1ª, perto de 70).

**Antes de rodar:** qual coluna vai mudar mais a chance dele?

In [ ]:
# Um passageiro inventado: troquem os valores
inventado = pd.DataFrame([{
    'pclass': 3,     # classe: 1, 2 ou 3
    'sex': 0,        # 1 = mulher, 0 = homem
    'age': 30,       # idade
    'sibsp': 0,      # irmãos/cônjuge a bordo
    'parch': 0,      # pais/filhos a bordo
    'fare': 8.0,     # tarifa
}])[COLUNAS]

explica(inventado)

---

# Parte 4 — O banco sem a idade

## Etapa 9 — Um modelo que nunca viu a idade

O banco das Aulas 3 e 5, as seis colunas da pessoa (ocupação, estado civil, escolaridade, e as três de crédito).
**A idade não entra.** Mesma divisão das Aulas 3 e 5 (`test_size=0.2`, `random_state=42`, `stratify=y`).

A regressão logística dá a cada pessoa uma **chance de dizer sim** (Aula 3). Vamos ver essa chance por faixa de idade,
usando a idade que o modelo **nunca viu**.

**Antes de rodar:** as três faixas vão receber chances parecidas?

In [ ]:
from sklearn.linear_model import LogisticRegression

URL_BANCO = "https://raw.githubusercontent.com/HegdeChaitra/Bank-Marketing-Campaign-Analysis/master/bank-additional-full.csv"
banco = pd.read_csv(URL_BANCO, sep=';')

# As seis colunas da pessoa, em 0 e 1 (Aula 3), sem a coluna repetida. Nenhuma delas é a idade.
PESSOA = ['job', 'marital', 'education', 'default', 'housing', 'loan']
X_pessoa = pd.get_dummies(banco[PESSOA], drop_first=True).drop(columns='loan_unknown')
y_banco = (banco['y'] == 'yes').astype(int)

# A idade vai junto na divisão só para a conferência depois: o modelo não a recebe
Xb_train, Xb_test, yb_train, yb_test, idade_train, idade_test = train_test_split(
    X_pessoa, y_banco, banco['age'], test_size=0.2, random_state=42, stratify=y_banco)

logistica = LogisticRegression(max_iter=1000)
logistica.fit(Xb_train, yb_train)

# A chance de 'sim' que o modelo dá a cada pessoa do teste, de 0 a 100
chance = 100 * logistica.predict_proba(Xb_test)[:, 1]

faixa = pd.cut(idade_test, [0, 29, 59, 200], labels=['menos de 30', '30 a 59', '60 ou mais'])
pd.DataFrame({'chance de sim (de cada 100)': chance, 'faixa': faixa.values}).groupby('faixa', observed=True).agg(
    pessoas=('chance de sim (de cada 100)', 'size'),
    chance_media=('chance de sim (de cada 100)', 'mean')).round(1)

O modelo nunca viu a idade. **Por onde ela entrou?** Cruzem a ocupação com a faixa de 60 anos ou mais,
numa tabela de contagens (Aula 3: `03_Avaliacao…`, `pd.crosstab`).

In [ ]:
# Linhas: a ocupação (banco['job']); colunas: 60 anos ou mais? (banco['age'] >= 60)
pd.crosstab(____, ____)

Para quem fala de uma coluna retirada no relatório, o teste é este: **dá para prever a coluna retirada a partir das outras?**
Se dá, ela ainda está lá dentro. Isso vai na **Seção 6** (*Restrições de uso*): tirar a coluna não basta para dizer que o modelo não usa aquilo.

---

# A célula para o projeto de vocês

**O que o modelo de vocês usou** (pergunta 6 do banco; Seções 3 e 5 do relatório).
Serve para qualquer modelo já treinado: árvore, floresta, logística, kNN, com ou sem `make_pipeline`.

- `modelo`: o modelo já treinado (`.fit` feito);
- `X_teste`, `y_teste`: o **teste** de vocês (a importância se mede nos dados que o modelo não viu);
- `metrica`: deixe `None` para usar a nota padrão do modelo (acurácia na classificação, R² na regressão),
  ou use a mesma da tabela de comparação (por exemplo `'recall'`, ou `'neg_mean_absolute_error'` na regressão).

Lembrem ao ler o resultado:
1. **Colunas que dizem o mesmo dividem a importância**: se duas colunas parecidas aparecem baixas, testem embaralhar as duas juntas;
2. **Usar não é causar**: a tabela diz o que o modelo usou, não o que causa o resultado no mundo;
3. **Uma coluna retirada pode continuar lá dentro** (proxy): se vocês tiraram uma coluna sensível (raça, sexo, idade, origem), testem se as outras colunas conseguem prevê-la.

In [ ]:
# ===================== O que o modelo usou (pergunta 6) =====================
import pandas as pd
from sklearn.inspection import permutation_importance

def o_que_o_modelo_usou(modelo, X_teste, y_teste, metrica=None, repeticoes=10):
    """Embaralha cada coluna do teste `repeticoes` vezes e mede quanto a métrica cai."""
    r = permutation_importance(modelo, X_teste, y_teste, scoring=metrica,
                               n_repeats=repeticoes, random_state=42)
    fator = 100 if metrica in (None, 'accuracy', 'recall', 'precision', 'f1') else 1
    return pd.DataFrame({
        'perde (média)': (fator * r.importances_mean).round(2),
        'pior vez': (fator * r.importances.max(axis=1)).round(2),
        'melhor vez': (fator * r.importances.min(axis=1)).round(2),
    }, index=X_teste.columns).sort_values('perde (média)', ascending=False)

# Exemplo, com a floresta da Parte 1 (troquem pelo modelo e pelo teste de vocês):
o_que_o_modelo_usou(floresta, X_test, y_test)